<a href="https://colab.research.google.com/github/qwuae1/MachineLearning/blob/master/MIMIC/Notebook/MIMIC_demo_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 30 Days Readmission Predicition

## 1 Overview and Background
This part is going to predicting whether a patient discharged from an intensive care unit (ICU) is likely to be readmitted within 30 days.

## 2 Preprocessing

+ Genrate lables: For each patient, we computed the intervals between his two sequent admissions. If the interval is shorter than 30 days, the readmission label for the encounter is 1; otherwise the readmission label is 0.


+ Create feature matrix: We joined the diagnosis table and the ICD code table to obtain the diagnosis for each encounter. Since the diagnosis are categorical variables, we converted the categorical variable into dummy variables. We also added some other predictive features, such as genders and admission types.


In [174]:
import sqlite3 
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import BaggingClassifier

from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import confusion_matrix

import time
import math

import warnings
warnings.filterwarnings('ignore')

from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
plt.rcParams['figure.dpi'] = 300


Here is the instruction to upload the database file.
Click on the folder icon.
![title](https://github.com/qwuae1/MachineLearning/blob/master/MIMIC/upload2.png?raw=true)

Click on the upload icon.

![title](https://github.com/qwuae1/MachineLearning/blob/master/MIMIC/upload1.png?raw=true)

In [175]:
conn = sqlite3.connect('/content/mimicdata.sqlite')
pd.set_option('display.max_columns', None)
table_df = pd.read_sql("SELECT * FROM sqlite_master where type='table'", conn)
table_df

,type,name,tbl_name,rootpage,sql
0,table,CHARTEVENTS,CHARTEVENTS,5,CREATE TABLE CHARTEVENTS\n (\tROW_ID INT NOT...
1,table,DIAGNOSES_ICD,DIAGNOSES_ICD,7,CREATE TABLE DIAGNOSES_ICD\n (\tROW_ID INT N...
2,table,D_ICD_DIAGNOSES,D_ICD_DIAGNOSES,9,CREATE TABLE D_ICD_DIAGNOSES\n (\tROW_ID INT...
3,table,D_ITEMS,D_ITEMS,12,CREATE TABLE D_ITEMS\n (\tROW_ID INT NOT NUL...
4,table,D_LABITEMS,D_LABITEMS,15,CREATE TABLE D_LABITEMS\n (\tROW_ID INT NOT ...
5,table,ICUSTAYS,ICUSTAYS,18,CREATE TABLE ICUSTAYS\n (\tROW_ID INT NOT NU...
6,table,INPUTEVENTS_MV,INPUTEVENTS_MV,21,CREATE TABLE INPUTEVENTS_MV\n (\tROW_ID INT ...
7,table,LABEVENTS,LABEVENTS,23,CREATE TABLE LABEVENTS\n (\tROW_ID INT NOT N...
8,table,OUTPUTEVENTS,OUTPUTEVENTS,27,CREATE TABLE OUTPUTEVENTS\n (\tROW_ID INT NO...
9,table,PRESCRIPTIONS,PRESCRIPTIONS,32,CREATE TABLE PRESCRIPTIONS\n (\tROW_ID INT N...


In [176]:
def display_table(table_name):
    print(table_name)
    display(pd.read_sql('SELECT * FROM {} LIMIT 5'.format(table_name), conn))

In [178]:
table_list = table_df['tbl_name'].values
for table_name in table_list:
  display_table(table_name)

CHARTEVENTS


,ROW_ID,SUBJECT_ID,HADM_ID,ICUSTAY_ID,ITEMID,CHARTTIME,STORETIME,CGID,VALUE,VALUENUM,UOM,WARNING,ERROR,RESULTSTATUS,STOPPED
0,5145594,40004,121157,256307,226707,2171-03-02 22:06:00,2171-03-02 22:06:00,16818,66,66.0,Inch,0,0,,
1,5145595,40004,121157,256307,226730,2171-03-02 22:06:00,2171-03-02 22:06:00,16818,168,168.0,cm,0,0,,
2,5145596,40004,121157,256307,220045,2171-03-02 23:00:00,2171-03-02 23:12:00,16818,57,57.0,bpm,0,0,,
3,5145597,40004,121157,256307,220179,2171-03-02 23:00:00,2171-03-02 23:12:00,16818,113,113.0,mmHg,0,0,,
4,5145598,40004,121157,256307,220180,2171-03-02 23:00:00,2171-03-02 23:12:00,16818,67,67.0,mmHg,0,0,,


DIAGNOSES_ICD


,ROW_ID,SUBJECT_ID,HADM_ID,SEQ_NUM,ICD9_CODE
0,379265,40004,121157,1,34591
1,379266,40004,121157,2,2762
2,379267,40004,121157,3,78604
3,379268,40004,121157,4,78060
4,379269,40004,121157,5,30470


D_ICD_DIAGNOSES


...

,ROW_ID,SUBJECT_ID,HADM_ID,ADMITTIME,DISCHTIME,DEATHTIME,ADMISSION_TYPE,ADMISSION_LOCATION,DISCHARGE_LOCATION,INSURANCE,LANGUAGE,RELIGION,MARITAL_STATUS,ETHNICITY,EDREGTIME,EDTIMEOUT,DIAGNOSIS,HOSPITAL_EXPIRE_FLAG,HAS_IOEVENTS_DATA,HAS_CHARTEVENTS_DATA
0,39793,40036,198489,2141-08-01 23:46:00,2141-08-09 19:15:00,,EMERGENCY,CLINIC REFERRAL/PREMATURE,HOME HEALTH CARE,Medicare,ENGL,CATHOLIC,MARRIED,WHITE,2141-08-01 19:03:00,2141-08-02 01:13:00,SEPSIS,0,1,1
1,39807,40080,162107,2106-05-31 16:43:00,2106-06-05 01:18:00,,EMERGENCY,CLINIC REFERRAL/PREMATURE,SNF,Medicaid,HAIT,UNOBTAINABLE,WIDOWED,BLACK/AFRICAN AMERICAN,2106-05-31 12:21:00,2106-05-31 17:21:00,CONGESTIVE HEART FAILURE,0,1,1
2,39809,40084,195762,2173-01-31 22:11:00,2173-02-05 01:31:00,2173-02-05 01:31:00,EMERGENCY,CLINIC REFERRAL/PREMATURE,DEAD/EXPIRED,Private,ENGL,CATHOLIC,SINGLE,WHITE,2173-01-31 20:47:00,2173-02-01 00:57:00,INTRACRANIAL HEMORRHAGE;OPEN FX,1,1,1
3,39817,40116,157106,2150-02-19 00:12:00,2150-03-11 13:58:00,,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,SHORT TERM HOSPITAL,Medicaid,ENGL,CATHOLIC,SINGLE,WHITE,,,GASTROINTESTINAL BLEED,0,1,1
4,39818,40120,146466,2120-01-27 20:41:00,2120-02-12 17:14:00,,EMERGENCY,EMERGENCY ROOM ADMIT,LONG TERM CARE HOSPITAL,Medicare,ENGL,PROTESTANT QUAKER,WIDOWED,BLACK/AFRICAN AMERICAN,2120-01-27 16:58:00,2120-01-27 21:43:00,CONGESTIVE HEART FAILURE,0,1,1


MICROBIOLOGYEVENTS


,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,SPEC_ITEMID,SPEC_TYPE_DESC,ORG_ITEMID,ORG_NAME,ISOLATE_NUM,AB_ITEMID,AB_NAME,DILUTION_TEXT,DILUTION_COMPARISON,DILUTION_VALUE,INTERPRETATION
0,50377,40120,146466,2120-02-03 00:00:00,2120-02-03 05:32:00,70064,STOOL,80139,CLOSTRIDIUM DIFFICILE,1,NaN,,,,NaN,
1,268038,40161,180396,2181-10-24 00:00:00,2181-10-24 18:26:00,70079,URINE,80002,ESCHERICHIA COLI,1,90012.0,GENTAMICIN,<=1,<=,1.0,S
2,268039,40161,180396,2181-10-24 00:00:00,2181-10-24 18:26:00,70079,URINE,80002,ESCHERICHIA COLI,1,90029.0,MEROPENEM,<=0.25,<=,0.0,S
3,268040,40161,180396,2181-10-24 00:00:00,2181-10-24 18:26:00,70079,URINE,80002,ESCHERICHIA COLI,1,90028.0,CEFEPIME,<=1,<=,1.0,S
4,268041,40161,180396,2181-10-24 00:00:00,2181-10-24 18:26:00,70079,URINE,80002,ESCHERICHIA COLI,1,90022.0,AMPICILLIN/SULBACTAM,4,=,4.0,S


PATIENTS


,ROW_ID,SUBJECT_ID,GENDER,DOB,DOD,DOD_HOSP,DOD_SSN,EXPIRE_FLAG
0,30802,40004,M,2118-09-11 00:00:00,2172-11-12 00:00:00,,2172-11-12 00:00:00,1
1,30810,40036,M,2075-04-25 00:00:00,2143-09-03 00:00:00,2143-09-03 00:00:00,2143-09-03 00:00:00,1
2,30821,40080,F,2027-08-04 00:00:00,2106-06-14 00:00:00,,2106-06-14 00:00:00,1
3,30823,40084,M,2149-10-04 00:00:00,2173-02-05 00:00:00,2173-02-05 00:00:00,2173-02-05 00:00:00,1
4,30828,40116,M,2109-03-21 00:00:00,2150-09-26 00:00:00,,2150-09-26 00:00:00,1


### 2.1 Create Labels

In [ ]:
query='''SELECT * FROM
                           (SELECT SUBJECT_ID, HADM_ID, ADMITTIME, DISCHTIME, 
                                   RANK() OVER(PARTITION BY SUBJECT_ID ORDER BY ADMITTIME) AS ADMITTIME_RANK 
                            FROM ADMISSIONS) a
                            LEFT JOIN 
                           (SELECT SUBJECT_ID, MAX(ADMITTIME_RANK) AS MAX_ADMITTIME_RANK FROM
                           (SELECT SUBJECT_ID, HADM_ID, ADMITTIME, DISCHTIME, 
                                   RANK() OVER(PARTITION BY SUBJECT_ID ORDER BY ADMITTIME) AS ADMITTIME_RANK 
                            FROM ADMISSIONS)
                            GROUP BY SUBJECT_ID) b
                            ON a.SUBJECT_ID = b.SUBJECT_ID'''
#　readmit_df = pd.read_sql_query(query,conn)
readmit_df = pd.read_csv('https://raw.githubusercontent.com/qwuae1/MachineLearning/master/MIMIC/intro_to_mimic/df.csv')
readmit_df[['ADMITTIME','DISCHTIME']] = readmit_df[['ADMITTIME','DISCHTIME']].apply(lambda x: pd.to_datetime(x, format='%Y-%m-%d %H:%M:%S'))

next_admittime = readmit_df['ADMITTIME'].values[1:]
next_admittime = np.append(next_admittime, next_admittime[0])
readmit_df['NEXT_ADMITTIME'] = next_admittime

readmit_df['30_READMIT'] = -1

readmit_df.loc[readmit_df['ADMITTIME_RANK'] == readmit_df['MAX_ADMITTIME_RANK'], '30_READMIT'] = 0

readmit_sub_df = readmit_df[readmit_df['30_READMIT']==-1]
interval = (readmit_sub_df['NEXT_ADMITTIME'] - readmit_sub_df['DISCHTIME']).dt.days.values
readmit_df.loc[readmit_df['30_READMIT']==-1, 'INTERVAL'] = interval 

readmit_df.loc[readmit_df['INTERVAL']<=30, '30_READMIT'] = 1
readmit_df.loc[readmit_df['INTERVAL']>30, '30_READMIT'] = 0

In [ ]:
readmit_df.head()

In [ ]:
readmit_label_df = readmit_df[['HADM_ID', '30_READMIT']]

In [ ]:
readmit_label_df.head()

### 2.2 Feature Matrix

In [ ]:
feat_sub1_df = pd.read_sql('''SELECT a.HADM_ID, b.SHORT_TITLE
                         FROM DIAGNOSES_ICD a
                         INNER JOIN D_ICD_DIAGNOSES b on a.ICD9_CODE = b.ICD9_CODE
                            
''',conn)

In [ ]:
feat_sub1_df = feat_sub1_df.groupby(['HADM_ID', 'SHORT_TITLE']).size().unstack()

In [ ]:
feat_sub1_df = feat_sub1_df.notnull()
feat_sub1_df = feat_sub1_df * 1
feat_sub1_df = feat_sub1_df.apply(pd.to_numeric)

In [ ]:
feat_sub1_df.head()

In [ ]:
feat_sub1_df.shape

In [ ]:
feat_sub2_df = pd.read_sql('''SELECT a.HADM_ID, a.ADMISSION_TYPE, b.gender 
                              FROM ADMISSIONS a 
                              INNER JOIN PATIENTS b on a.SUBJECT_ID = b.SUBJECT_ID''',conn)

In [ ]:
feat_sub2_df = pd.get_dummies(feat_sub2_df)

In [ ]:
feat_sub2_df.head()

In [ ]:
feat_df = pd.merge(feat_sub1_df, feat_sub2_df, left_on=['HADM_ID'], right_on=['HADM_ID'], how='inner')

In [ ]:
feat_df.head()

### 2.3 Design Matrix

In [ ]:
design_matrix = pd.merge(readmit_label_df, feat_df, left_on =['HADM_ID'], right_on=['HADM_ID'], how='inner')

In [ ]:
design_matrix.head()

## 3 Data Exploration

We visualized the readmission proportion of the two datasets. Apparently, the datasets have highly imbalanced classes.

In [ ]:
X = design_matrix.iloc[:, 2:].values

In [ ]:
X

In [ ]:
print('number of instance : the number features = {}'.format(X.shape[0] / X.shape[1]) )

In [ ]:
y = design_matrix['30_READMIT'].values

In [ ]:
H1_num = len(y[y==1])
H0_num = len(y[y==0])
total_num = len(y)

In [ ]:
print('number of data instance in class1:{}'.format(H1_num))

In [ ]:
labels = '30Days Readmission-Y', '30Days Readmission-N'
sizes = [H1_num/total_num*100, H0_num/total_num*100]
colors = ['pink', 'skyblue']
explode = (.05, 0)
fig = plt.figure(figsize=(3,3))
plt.pie(sizes, explode=explode, colors=colors, autopct='%1.1f%%', startangle=90)
plt.legend(labels, fontsize='small', loc='center left', bbox_to_anchor=(1, 0, 0.5, 1))

## 4 Models

### 4.1 Logistic Regression
### 4.1.1 Oversampling

In [ ]:
ros = RandomOverSampler(random_state=42)

skf = StratifiedKFold(n_splits=5, shuffle=False, random_state=0)
y_true = []
y_pred = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)
    
    lr_clf = LogisticRegression(penalty='l1', random_state=0, solver='liblinear') # l1-based feature selection
    lr_clf.fit(X_train_ros, y_train_ros)
    y_pred_sub = lr_clf.predict(X_test)
    y_true = np.append(y_true, y_test)
    y_pred = np.append(y_pred, y_pred_sub)
    
lr_os_conf = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = lr_os_conf.ravel()
lr_os_sen = tp/(tp+fn)
lr_os_spe = tn/(tn+fp)
lr_os_prec = tp/(tp+fp)
lr_os_acc = (tp+tn)/(tp+tn+fp+fn)

print ('Logistic Regression, Over_Sampling')
print()
print('confusion matrix:')
print(lr_os_conf)
print('accuracy score:{:.4}'.format(lr_os_acc))

### 4.1.2 Undersampling

In [ ]:
rus = RandomUnderSampler(random_state=42)

skf = StratifiedKFold(n_splits=5, shuffle=False, random_state=0)
y_true = []
y_pred = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)
    
    lr_clf = LogisticRegression(penalty='l1', random_state=0, solver='liblinear') # l1-based feature selection
    lr_clf.fit(X_train_rus, y_train_rus)
    y_pred_sub = lr_clf.predict(X_test)
    y_true = np.append(y_true, y_test)
    y_pred = np.append(y_pred, y_pred_sub)
    
lr_us_conf = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = lr_us_conf.ravel()
lr_us_sen = tp/(tp+fn)
lr_us_spe = tn/(tn+fp)
lr_us_prec = tp/(tp+fp)
lr_us_acc = (tp+tn)/(tp+tn+fp+fn)

print ('Logistic Regression, Under_Sampling')
print()
print('confusion matrix:')
print(lr_us_conf)
print('accuracy score:{:.4}'.format(lr_us_acc))

### 4.2 Bagging Logistic Regression

### 4.2.2 Oversampling

In [ ]:
ros = RandomOverSampler(random_state=42)

skf = StratifiedKFold(n_splits=5, shuffle=False, random_state=0)
y_true = []
y_pred = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)
    
    base_estimator = LogisticRegression(penalty='l1', random_state=0, solver='liblinear') # l1-based feature selection
    max_features = int(math.sqrt(X_train.shape[1]))
    blr_clf = BaggingClassifier(base_estimator=base_estimator, n_estimators=100, max_features=max_features, random_state=0)
    blr_clf.fit(X_train_ros, y_train_ros)
    y_pred_sub = blr_clf.predict(X_test)
    y_true = np.append(y_true, y_test)
    y_pred = np.append(y_pred, y_pred_sub)
    
blr_os_conf = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = blr_os_conf.ravel()
blr_os_sen = tp/(tp+fn)
blr_os_spe = tn/(tn+fp)
blr_os_prec = tp/(tp+fp)
blr_os_acc = (tp+tn)/(tp+tn+fp+fn)

print ('Bagging Logistic Regression, Over_Sampling')
print()
print('confusion matrix:')
print(blr_os_conf)
print('accuracy score:{:.4}'.format(blr_os_acc))

### 4.2.2 Undersampling

In [ ]:
rus = RandomUnderSampler(random_state=42)

skf = StratifiedKFold(n_splits=5, shuffle=False, random_state=0)
y_true = []
y_pred = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)
    
    base_estimator = LogisticRegression(penalty='l1', random_state=0, solver='liblinear') # l1-based feature selection
    max_features = int(math.sqrt(X_train.shape[1]))
    blr_clf = BaggingClassifier(base_estimator=base_estimator, n_estimators=100, max_features=max_features, random_state=0)
    blr_clf.fit(X_train_rus, y_train_rus)
    y_pred_sub = blr_clf.predict(X_test)
    y_true = np.append(y_true, y_test)
    y_pred = np.append(y_pred, y_pred_sub)
    
blr_us_conf = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = blr_us_conf.ravel()
blr_us_sen = tp/(tp+fn)
blr_us_spe = tn/(tn+fp)
blr_us_prec = tp/(tp+fp)
blr_us_acc = (tp+tn)/(tp+tn+fp+fn)

print ('Bagging Logistic Regression, Under_Sampling')
print()
print('confusion matrix:')
print(blr_us_conf)
print('accuracy score:{:.4}'.format(blr_us_acc))

### 4.3 Decision Tree

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=False, random_state=0)
y_true = []
y_pred = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # l1-based feature selection
    logit_clf = LogisticRegression(penalty='l1', random_state=0, solver='liblinear')
    logit_clf.fit(X_train, y_train)
    select_model = SelectFromModel(logit_clf)
    select_model.fit(X_train, y_train)
    X_train_new = select_model.transform(X_train)
    X_test_new = select_model.transform(X_test)
    
    dt_clf = DecisionTreeClassifier(max_depth=3, random_state=0, class_weight='balanced')
    dt_clf.fit(X_train_new, y_train)
    y_pred_sub = dt_clf.predict(X_test_new)
    y_true = np.append(y_true, y_test)
    y_pred = np.append(y_pred, y_pred_sub)

dt_conf = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = dt_conf.ravel()
dt_sen = tp/(tp+fn)
dt_spe = tn/(tn+fp)
dt_prec = tp/(tp+fp)
dt_acc = (tp+tn)/(tp+tn+fp+fn)

print ('Decision Tree')
print()
print('confusion matrix:')
print(dt_conf)
print('accuracy score:{:.4}'.format(dt_acc))

### 4.4 Random Forest

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=False, random_state=0)
y_true = []
y_pred = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # l1-based feature selection
    logit_clf = LogisticRegression(penalty='l1', random_state=0, solver='liblinear')
    logit_clf.fit(X_train, y_train)
    select_model = SelectFromModel(logit_clf)
    select_model.fit(X_train, y_train)
    X_train_new = select_model.transform(X_train)
    X_test_new = select_model.transform(X_test)
    
    rf_clf = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=0, class_weight='balanced')
    rf_clf.fit(X_train_new, y_train)
    y_pred_sub = rf_clf.predict(X_test_new)
    y_true = np.append(y_true, y_test)
    y_pred = np.append(y_pred, y_pred_sub)

rf_conf = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = rf_conf.ravel()
rf_sen = tp/(tp+fn)
rf_spe = tn/(tn+fp)
rf_prec = tp/(tp+fp)
rf_acc = (tp+tn)/(tp+tn+fp+fn)

print ('Random Forest')
print()
print('confusion matrix:')
print(rf_conf)
print('accuracy score:{:.4}'.format(rf_acc))

### 4.5 Neural Network


In [ ]:
from keras import backend as K
from keras.models import Sequential
from keras.layers import Activation, Dense, Dropout
seed_value= 0
# 1. Set `PYTHONHASHSEED` environment variable at a fixed value
import os
os.environ['PYTHONHASHSEED']=str(seed_value)
# 2. Set `python` built-in pseudo-random generator at a fixed value
import random
random.seed(seed_value)
# 3. Set `numpy` pseudo-random generator at a fixed value
np.random.seed(seed_value)
# 4. Set `tensorflow` pseudo-random generator at a fixed value
import tensorflow as tf
tf.random.set_seed(seed_value)
os.environ['KMP_DUPLICATE_LIB_OK']='True'


In [ ]:
rus = RandomUnderSampler(random_state=42)

skf = StratifiedKFold(n_splits=5, shuffle=False, random_state=0)
y_true = []
y_pred = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)
    
    model = Sequential()
    model.add(Dense(512, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(8, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    class_weight = class_weight = {0: 1., 1: 1.}
    model.fit(X_train_rus, y_train_rus, batch_size=500, epochs=20, validation_data=(X_test, y_test), verbose=1, class_weight=class_weight)
    y_pred_sub = model.predict(X_test).flatten()
    y_pred_sub[y_pred_sub >= 0.5] = 1
    y_pred_sub[y_pred_sub < 0.5] = 0
    y_true = np.append(y_true, y_test)
    y_pred = np.append(y_pred, y_pred_sub)
    
nn_us_conf = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = nn_us_conf.ravel()
nn_us_sen = tp/(tp+fn)
nn_us_spe = tn/(tn+fp)
nn_us_prec = tp/(tp+fp)
nn_us_acc = (tp+tn)/(tp+tn+fp+fn)

print ('Neural Network(Deep Learning)')
print()
print(nn_us_conf)

## 5 Comparision

The classes are highly imbalanced!

### 5.1 Confusion Matrix

In [ ]:
def draw_conf_mat(conf_mat, title, ax):
    conf_plot = ax.matshow(conf_mat, cmap=plt.cm.Blues)
    class_names = ['Not', 'Readmit']
    ax.set_xticklabels([''] + class_names,fontdict={'fontsize':'5'})
    ax.set_yticklabels([''] + class_names,fontdict={'fontsize':'5'})
    plt.title(title, fontsize='5')
    plt.xlabel('Prediction', fontsize='5')
    plt.ylabel('Truth', fontsize='5')
    num = conf_mat.ravel()
    plt.text(-.15, 0, str(num[0]), fontdict={'color': 'white', 'size':5})
    plt.text(.85, 0, str(num[1]), fontdict={'color': 'black', 'size':5})
    plt.text(-.15, 1, str(num[2]), fontdict={'size':5})
    plt.text(.85, 1, str(num[3]), fontdict={'size':5})

In [ ]:
conf_mat_list = [lr_os_conf, lr_us_conf, blr_os_conf,blr_us_conf,dt_conf, rf_conf, nn_us_conf]
clf = ['LR OverSampling', 'LR UnderSampling', 
               'Bagging LR OverSampling', 'Bagging LR UnderSampling', 
               'Decision Tree', 'Random Forest', 'Neural Network', 'Neural Network UnderSampling']

In [ ]:
fig = plt.figure(figsize=(5,5))
for i in range(1, 5):
    ax = fig.add_subplot(int('22{}'.format(i)))
    draw_conf_mat(conf_mat_list[i-1], clf[i-1], ax)
    if i!=1 and i!=3:
        ax.set_ylabel('')


In [ ]:
#%%
fig = plt.figure(figsize=(5,5))
for i in range(5, 7):
    ax = fig.add_subplot(int('22{}'.format(i-4)))
    draw_conf_mat(conf_mat_list[i-1], clf[i-1], ax)
    if i!=5 and i!=7:
        ax.set_ylabel('')